# OmniLingual ASR 1B Levantine Custom Run
This notebook mirrors the Levantine Whisper run flow using the official OmniLingual 1B LLM-ASR recipe, while reusing the exact same source train/val/test split manifests.

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path('/home/MohammadNabulsi/whisper')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import (
    NOTEBOOK_PATH,
    SMOKE_NOTEBOOK_PATH,
    config_snapshot,
    ensure_run_layout,
    make_config,
    setup_logging,
)

RUN_SMOKE_TEST = True
EVAL_SAMPLE_CAP = 1
TRAIN_NUM_STEPS = 1
RUN_BASELINE_BEFORE_TRAIN = True
RUN_POST_TRAIN_EVAL = True

config = make_config(
    smoke_mode=RUN_SMOKE_TEST,
    eval_sample_cap=EVAL_SAMPLE_CAP,
    train_num_steps=TRAIN_NUM_STEPS,
    run_baseline_before_train=RUN_BASELINE_BEFORE_TRAIN,
    run_post_train_eval=RUN_POST_TRAIN_EVAL,
)

ensure_run_layout()
log_path = setup_logging()
print(json.dumps(config_snapshot(config), ensure_ascii=False, indent=2))
print('Notebook path:', SMOKE_NOTEBOOK_PATH if RUN_SMOKE_TEST else NOTEBOOK_PATH)
print('Log path:', log_path)


{
  "run_id": "omnilingual_asr_1b_levantine_custom_streaming_5minckpt",
  "model_card": "omniASR_LLM_1B_v2",
  "model_family": "wav2vec2_llama",
  "model_arch": "1b_v2",
  "tokenizer_name": "omniASR_tokenizer_written_v2",
  "language": "apc_Arab",
  "sample_rate": 16000,
  "min_audio_seconds": 0.3,
  "max_audio_seconds": 30.0,
  "smoke_mode": true,
  "eval_sample_cap": 1,
  "run_baseline_before_train": true,
  "run_post_train_eval": true,
  "train_num_steps": 1,
  "validate_every_n_steps": 1,
  "checkpoint_every_n_steps": 1,
  "publish_metrics_every_n_steps": 1,
  "batch_size": 1,
  "max_num_elements": 480000,
  "beta_corpus": 0.5,
  "beta_language": 0.5,
  "freeze_encoder_for_n_steps": 0,
  "learning_rate": 5e-05,
  "run_dir": "/home/MohammadNabulsi/whisper/Runs/omnilingual_asr_1b_levantine_custom_streaming_5minckpt",
  "source_run_dir": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt",
  "dataset_dir": "/home/MohammadNabulsi/whisper/Runs/omnilin

In [2]:
from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import prepare_dataset

manifest_state = prepare_dataset(config)
selection_summary = manifest_state['selection_summary']
print(json.dumps(selection_summary, ensure_ascii=False, indent=2))


{
  "run_id": "omnilingual_asr_1b_levantine_custom_streaming_5minckpt",
  "source_manifests": {
    "train": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/manifests/train/manifest_train_custom_levantine_lt30s.jsonl",
    "val": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/manifests/val/manifest_val_custom_levantine_lt30s.jsonl",
    "test": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/manifests/test/manifest_test_custom_levantine_lt30s.jsonl"
  },
  "selected_counts": {
    "train": 1,
    "val": 1,
    "test": 1
  },
  "selected_hours": {
    "train": 0.006408061342592593,
    "val": 0.0016013015716666668,
    "test": 0.0010722985986111112
  },
  "dataset_dir": "/home/MohammadNabulsi/whisper/Runs/omnilingual_asr_1b_levantine_custom_streaming_5minckpt/dataset/version=0",
  "dataset_summary_path": "/home/MohammadNabulsi/whisper/Runs/omnilingual_asr_1b_levantine_cu

In [3]:
from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import run_predictions

baseline_test_metrics = None
if config.run_baseline_before_train:
    baseline_test_metrics = run_predictions(
        manifest_state['test_rows'],
        config,
        model_card=config.model_card,
        name='baseline_test_predictions',
    )
print(json.dumps(baseline_test_metrics, ensure_ascii=False, indent=2))


Output()

/home/MohammadNabulsi/whisper/third_party/omnilingual-asr/src/omnilingual_asr/models/inference/pipeline.py:442: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "waveform": torch.tensor(x["waveform"]),


{
  "num_predictions": 1,
  "prediction_path": "/home/MohammadNabulsi/whisper/Runs/omnilingual_asr_1b_levantine_custom_streaming_5minckpt/eval_predictions_smoke/baseline_test_predictions.jsonl",
  "total_hours": 0.0010722985986111112,
  "wer": 0.6,
  "cer": 0.5208333333333334,
  "wer_loose": 0.6,
  "cer_loose": 0.5208333333333334,
  "wer_no_punct": 0.6,
  "cer_no_punct": 0.5208333333333334
}


In [4]:
from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import run_training

training_summary = run_training(config, manifest_state)
print(json.dumps(training_summary, ensure_ascii=False, indent=2))


2026-06-23 21:23:06,557 | INFO | omnilingual_asr_1b_run | Running training command: /home/MohammadNabulsi/whisper/.venv/bin/python -m workflows.recipes.wav2vec2.asr /home/MohammadNabulsi/whisper/Runs/omnilingual_asr_1b_levantine_custom_streaming_5minckpt/smoke_checkpoints --config-file /home/MohammadNabulsi/whisper/Runs/omnilingual_asr_1b_levantine_custom_streaming_5minckpt/configs/llm_finetune_smoke.yaml


2026-06-23 21:23:17 INFO     fairseq2 - No cluster detected.                    
                    INFO     fairseq2 - Log files are stored under              
                             /home/MohammadNabulsi/whisper/Runs/omnilingual_asr_
                             1b_levantine_custom_streaming_5minckpt/smoke_checkp
                             oints/ws_1.ffa2d5b6.                               
2026-06-23 21:23:18 INFO     fairseq2 - Config: {                               
                                 'model': {                                     
                                     'name': 'omniASR_LLM_1B_v2',               
                                     'path': None,                              
                                     'family': 'wav2vec2_asr',                  
                                     'arch': None,                              
                                     'config_overrides': None,                  
                            

                    INFO     fairseq2 - Random number generator seed set to 2.  
                    INFO     fairseq2 - Number of threads used for intra-op     
                             parallelism set to 12.                             


                    INFO     fairseq2 - Device of the process set to cuda:0.    
                    INFO     fairseq2 - Default SDPA variant set to torch.      
                    INFO     fairseq2 - Host - Name:                            
                             student-teaching-vm-2.europe-west4-a.c.ai-engine-44
                             8418.internal | Platform:                          
                             Linux-6.17.0-1008-gcp-x86_64-with-glibc2.39 | Linux
                             Distribution: Ubuntu 24.04.4 LTS | Number of CPUs: 
                             12/12 | Memory: 167.04 GiB                         
                    INFO     fairseq2 - Device - ID: cuda:0 | Name: NVIDIA      
                             A100-SXM4-80GB | Memory: 79.25 GiB | Number of SMs:
                             108 | Compute Capability: 8.0                      
                    INFO     fairseq2 - Software - Python: 3.12.3 | PyTorch:    
                            

                    INFO     fairseq2 - Creating the root gang.                 
                    INFO     fairseq2 - Root gang created.                      
                    INFO     fairseq2 - Creating parallel gangs.                
                    INFO     fairseq2 - Creating data parallel gang with 1      
                             process(es).                                       
                    INFO     fairseq2 - Data parallel gang created.             
                    INFO     fairseq2 - Creating tensor parallel gang with 1    
                             process(es).                                       
                    INFO     fairseq2 - Tensor parallel gang created.           
                    INFO     fairseq2 - Creating pipeline parallel gang with 1  
                             process(es).                                       
                    INFO     fairseq2 - Pipeline parallel gang created.         
                    INFO    

...

train: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━     1/1 100% 0:00:00

train: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━     1/1 100% 0:00:00

2026-06-23 21:23:54 INFO     fairseq2 - Checkpoint at step 1 saved.             
train: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━     1/1 100% 0:00:00


                    INFO     fairseq2 - Task finished in 36 second(s) after 1   
                             step(s)!                                           


{
  "backend": "fairseq2",
  "best_checkpoint": "/home/MohammadNabulsi/whisper/Runs/omnilingual_asr_1b_levantine_custom_streaming_5minckpt/smoke_checkpoints/ws_1.ffa2d5b6/checkpoints/step_1/model",
  "model_asset_name": "omnilingual_asr_1b_levantine_custom_streaming_5minckpt_smoke_checkpoint",
  "output_dir": "/home/MohammadNabulsi/whisper/Runs/omnilingual_asr_1b_levantine_custom_streaming_5minckpt/smoke_checkpoints",
  "completed_at": "2026-06-23T21:23:57Z",
  "train_rows": 1,
  "val_rows": 1
}


In [5]:
from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import model_asset_name, run_predictions

local_model_card = model_asset_name(config)
val_prediction_metrics = run_predictions(
    manifest_state['val_rows'],
    config,
    model_card=local_model_card,
    name='tuned_val_predictions',
) if config.run_post_train_eval else None

test_prediction_metrics = run_predictions(
    manifest_state['test_rows'],
    config,
    model_card=local_model_card,
    name='tuned_test_predictions',
) if config.run_post_train_eval else None

print('Validation metrics:')
print(json.dumps(val_prediction_metrics, ensure_ascii=False, indent=2))
print('Test metrics:')
print(json.dumps(test_prediction_metrics, ensure_ascii=False, indent=2))


parameter load:    ━━━━━━━━━━          ━━━━━━━━━━            0/919   0% -:--:--

parameter load:       ━━━━━━━━━━          ━━━━━━━━━━         0/919   0% -:--:--

parameter load:          ━━━━━━━━━━          ━━━━━━━━━━      0/919   0% -:--:--

parameter load: ━━          ━━━━━━━━━━          ━━━━━━━━     0/919   0% -:--:--

parameter load: ━━━━━          ━━━━━━━━━━          ━━━━━     0/919   0% -:--:--

...

parameter load: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   919/919 100% 0:00:00

parameter load: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   919/919 100% 0:00:00

parameter load: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   919/919 100% 0:00:00


/home/MohammadNabulsi/whisper/third_party/omnilingual-asr/src/omnilingual_asr/models/inference/pipeline.py:442: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "waveform": torch.tensor(x["waveform"]),


Validation metrics:
{
  "num_predictions": 1,
  "prediction_path": "/home/MohammadNabulsi/whisper/Runs/omnilingual_asr_1b_levantine_custom_streaming_5minckpt/eval_predictions_smoke/tuned_val_predictions.jsonl",
  "total_hours": 0.0016013015716666668,
  "wer": 0.3333333333333333,
  "cer": 0.13636363636363635,
  "wer_loose": 0.3333333333333333,
  "cer_loose": 0.13636363636363635,
  "wer_no_punct": 0.25,
  "cer_no_punct": 0.09523809523809523
}
Test metrics:
{
  "num_predictions": 1,
  "prediction_path": "/home/MohammadNabulsi/whisper/Runs/omnilingual_asr_1b_levantine_custom_streaming_5minckpt/eval_predictions_smoke/tuned_test_predictions.jsonl",
  "total_hours": 0.0010722985986111112,
  "wer": 0.6,
  "cer": 0.5208333333333334,
  "wer_loose": 0.6,
  "cer_loose": 0.5208333333333334,
  "wer_no_punct": 0.6,
  "cer_no_punct": 0.5208333333333334
}


In [6]:
from Runs.omnilingual_asr_1b_levantine_custom_streaming_5minckpt.pipeline import write_integrity_report, write_summary_report

summary_report = write_summary_report(
    config,
    selection_summary,
    baseline_test_metrics,
    val_prediction_metrics,
    test_prediction_metrics,
    training_summary,
)
integrity_report = write_integrity_report(config, training_summary, summary_report)
print(json.dumps(summary_report, ensure_ascii=False, indent=2))
print(json.dumps(integrity_report, ensure_ascii=False, indent=2))


{
  "run_id": "omnilingual_asr_1b_levantine_custom_streaming_5minckpt",
  "model_card": "omniASR_LLM_1B_v2",
  "selection_summary": {
    "run_id": "omnilingual_asr_1b_levantine_custom_streaming_5minckpt",
    "source_manifests": {
      "train": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/manifests/train/manifest_train_custom_levantine_lt30s.jsonl",
      "val": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/manifests/val/manifest_val_custom_levantine_lt30s.jsonl",
      "test": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/manifests/test/manifest_test_custom_levantine_lt30s.jsonl"
    },
    "selected_counts": {
      "train": 1,
      "val": 1,
      "test": 1
    },
    "selected_hours": {
      "train": 0.006408061342592593,
      "val": 0.0016013015716666668,
      "test": 0.0010722985986111112
    },
    "dataset_dir": "/home/MohammadNabulsi/whisper/Runs/om